# STATE model-size sweep on PBMC — loss curves & MI by model size

5 architectures spanning ~1.8M → ~96M transformer-body params, all trained on the largest PBMC size (100k cells) across 10 quality levels. Hyperparameters are held fixed at STATE defaults so the only knob varying is parameter count.

Plots overlay one curve per model size — no box plots.

In [13]:
# Walk the on-disk layout produced by the model-size sweep:
#   model_sizing/model_sizing_NN/<size>/<quality>/
#     ├─ config.json                                               (arch + metadata; written pre-train)
#     ├─ result.json                                               (post-run; status, train_time_s)
#     ├─ checkpoints/**/version_*/metrics.csv                      (Lightning train/val loss curves)
#     └─ MI/<seed>/Y_<signal>_<quality>/lmi_mutual_information.txt
#
# Per-step training and validation loss curves come from Lightning's CSVLogger.
# The DataFrame holds scalars + arch fields; raw curves live in CURVES.
import json
import re
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path('/home/igor/noise_scaling/data/other/model_sizing')
MI_SEED = 42  # first seed emitted by latentmi
ARCH_KEYS = ('emsize', 'd_hid', 'nhead', 'nlayers', 'output_dim', 'pad_length')

def _num(s: str):
    try: return int(s)
    except ValueError:
        try: return float(s)
        except ValueError: return s

def _read_metrics(leaf: Path):
    """Return (train_df, val_df) from Lightning's metrics.csv, or (None, None)."""
    matches = sorted(leaf.glob('checkpoints/**/version_*/metrics.csv'))
    if not matches:
        return None, None
    m = pd.read_csv(matches[0])
    tcol, vcol = 'trainer/train_loss', 'validation/val_loss'
    train_df = (m.dropna(subset=[tcol])[['step', tcol]].rename(columns={tcol: 'train_loss'})
                 .sort_values('step').reset_index(drop=True)) if tcol in m.columns else None
    val_df   = (m.dropna(subset=[vcol])[['step', vcol]].rename(columns={vcol: 'val_loss'})
                 .sort_values('step').reset_index(drop=True)) if vcol in m.columns else None
    return train_df, val_df

def _approx_params(arch: dict) -> float:
    """Approximate transformer-body params (excludes shared pe_embedding).
    Per layer ≈ 4*emsize^2 (attention) + 2*emsize*d_hid (FFN)."""
    e, h, L = arch.get('emsize', 0), arch.get('d_hid', 0), arch.get('nlayers', 0)
    return L * (4 * e * e + 2 * e * h)

rows = []
CURVES: dict = {}  # (trial_id, size, quality) -> {'train': DataFrame, 'val': DataFrame}

for trial_dir in sorted(OUTPUT_DIR.glob('model_sizing_*')):
    if not trial_dir.is_dir():
        continue
    m = re.match(r'model_sizing_(\d+)$', trial_dir.name)
    if not m:
        continue
    trial_id = int(m.group(1))

    for size_dir in sorted(p for p in trial_dir.iterdir() if p.is_dir()):
        size_val = _num(size_dir.name)
        if not isinstance(size_val, (int, float)):
            continue
        for q_dir in sorted(p for p in size_dir.iterdir() if p.is_dir()):
            quality_val = _num(q_dir.name)
            if not isinstance(quality_val, (int, float)):
                continue

            cfg_path = q_dir / 'config.json'
            cfg = json.loads(cfg_path.read_text()) if cfg_path.exists() else {}
            arch = cfg.get('arch') or {}
            row = {
                'trial_id': trial_id,
                'trial_name': cfg.get('trial_name', f'model_sizing_{trial_id:02d}'),
                'size': int(size_val),
                'quality': float(quality_val),
                **{k: arch.get(k) for k in ARCH_KEYS},
                'approx_params': _approx_params(arch),
            }

            res_path = q_dir / 'result.json'
            if res_path.exists():
                try:
                    res = json.loads(res_path.read_text())
                    row['status'] = res.get('status', 'unknown')
                    row['train_time_s'] = res.get('train_time_s')
                except json.JSONDecodeError:
                    row['status'] = 'corrupt'
            else:
                row['status'] = 'pending'

            train_df, val_df = _read_metrics(q_dir)
            if train_df is not None and len(train_df):
                row['train_loss_final'] = float(train_df['train_loss'].iloc[-1])
                row['n_train_steps'] = int(train_df['step'].iloc[-1])
            if val_df is not None and len(val_df):
                row['val_loss_final'] = float(val_df['val_loss'].iloc[-1])
            CURVES[(trial_id, int(size_val), float(quality_val))] = {
                'train': train_df, 'val': val_df,
            }

            mi_root = q_dir / 'MI' / str(MI_SEED)
            if mi_root.is_dir():
                for sig_dir in filter(Path.is_dir, mi_root.iterdir()):
                    sm = re.match(r'Y_(.+)_[0-9][0-9_.]*$', sig_dir.name)
                    signal = sm.group(1) if sm else sig_dir.name
                    f = sig_dir / 'lmi_mutual_information.txt'
                    if f.exists():
                        try: row[f'mi_{signal}'] = float(f.read_text().strip())
                        except ValueError: pass
            rows.append(row)

df = pd.DataFrame(rows)
if len(df):
    df = df.sort_values(['trial_id', 'size', 'quality']).reset_index(drop=True)
mi_cols_found = sorted(c for c in df.columns if c.startswith('mi_'))
n_trials = df['trial_id'].nunique() if len(df) else 0
n_ok = int((df['status'] == 'ok').sum()) if 'status' in df.columns else 0
n_curves = sum(1 for v in CURVES.values() if v['train'] is not None and len(v['train']))
print(f'{len(df)} rows across {n_trials} model-size configs  ok={n_ok}  curves={n_curves}  mi_cols={mi_cols_found}')

# Per-trial label: 'M0  e192/L4  (1.8M)'  — used in legends below.
TRIAL_LABEL = {}
for tid, g in df.groupby('trial_id'):
    r0 = g.iloc[0]
    TRIAL_LABEL[int(tid)] = (
        f"M{int(tid)}  e{int(r0['emsize'])}/L{int(r0['nlayers'])}  "
        f"({r0['approx_params']/1e6:.1f}M)"
    )
df

50 rows across 5 model-size configs  ok=3  curves=19  mi_cols=['mi_celltype.l3', 'mi_protein_counts']


,trial_id,trial_name,size,quality,emsize,d_hid,nhead,nlayers,output_dim,pad_length,approx_params,status,train_time_s,train_loss_final,n_train_steps,val_loss_final,mi_celltype.l3,mi_protein_counts
0,0,model_sizing_00_100000_0.0012346,100000,0.001235,192,384,3,4,192,2048,1179648,error,7.495939,12.544748,1369.0,9.729087,NaN,NaN
1,0,model_sizing_00_100000_0.0025982,100000,0.002598,192,384,3,4,192,2048,1179648,pending,NaN,NaN,NaN,NaN,NaN,NaN
2,0,model_sizing_00_100000_0.0054682,100000,0.005468,192,384,3,4,192,2048,1179648,error,7.060601,NaN,NaN,NaN,NaN,NaN
3,0,model_sizing_00_100000_0.0115083,100000,0.011508,192,384,3,4,192,2048,1179648,ok,3189.473726,15.027283,9999.0,14.883609,0.23749,0.22297
4,0,model_sizing_00_100000_0.02422,100000,0.024220,192,384,3,4,192,2048,1179648,ok,3300.543255,17.222097,8999.0,16.811266,0.06008,0.08055
5,0,model_sizing_00_100000_0.050973,100000,0.050973,192,384,3,4,192,2048,1179648,error,7.221223,NaN,NaN,NaN,NaN,NaN
6,0,model_sizing_00_100000_0.1072766,100000,0.107277,192,384,3,4,192,2048,1179648,pending,NaN,NaN,NaN,NaN,NaN,NaN
7,0,model_sizing_00_100000_0.225772,100000,0.225772,192,384,3,4,192,2048,1179648,pending,NaN,NaN,NaN,NaN,NaN,NaN
8,0,model_sizing_00_100000_0.4751547,100000,0.475155,192,384,3,4,192,2048,1179648,pending,NaN,NaN,NaN,NaN,NaN,NaN
9,0,model_sizing_00_100000_1.0,100000,1.000000,192,384,3,4,192,2048,1179648,ok,2851.870132,18.599567,6999.0,17.208160,1.16659,1.11046


In [ ]:
# Overlay the *original* STATE runs (default arch: emsize=256, d_hid=512, nhead=4, nlayers=3)
# from the production sweep at PBMC size=100k, across all available qualities.
# These come from `run_pbmc_whole.py` (algo/state.py) and live at:
#   data/PBMC/100000/<quality>/results/State/model/checkpoints/.../version_0/metrics.csv
# Registered as a pseudo-trial (trial_id = -1) so the existing plotting cells pick it up;
# downstream cells override its color/linestyle so it stands out from the sweep.
ORIGINAL_STATE_DIR = Path('/home/igor/noise_scaling/data/PBMC/100000')
ORIGINAL_STATE_ARCH = {  # algo/state.py defaults; verified against hparams.yaml
    'emsize': 256, 'd_hid': 512, 'nhead': 4,
    'nlayers': 3, 'output_dim': 256, 'pad_length': 2048,
}
ORIGINAL_TRIAL_ID = -1
_orig_params = _approx_params(ORIGINAL_STATE_ARCH)
print(f'Original STATE arch: emsize={ORIGINAL_STATE_ARCH["emsize"]} '
      f'd_hid={ORIGINAL_STATE_ARCH["d_hid"]} nlayers={ORIGINAL_STATE_ARCH["nlayers"]} '
      f'({_orig_params/1e6:.1f}M params)')

_orig_rows = []
for q_dir in sorted(p for p in ORIGINAL_STATE_DIR.iterdir() if p.is_dir()):
    quality_val = _num(q_dir.name)
    if not isinstance(quality_val, (int, float)):
        continue
    metrics_root = q_dir / 'results' / 'State' / 'model'
    train_df, val_df = _read_metrics(metrics_root)
    if train_df is None and val_df is None:
        continue
    row = {
        'trial_id': ORIGINAL_TRIAL_ID,
        'trial_name': 'original_state',
        'size': 100_000,
        'quality': float(quality_val),
        **ORIGINAL_STATE_ARCH,
        'approx_params': _orig_params,
        'status': 'ok',
    }
    if train_df is not None and len(train_df):
        row['train_loss_final'] = float(train_df['train_loss'].iloc[-1])
        row['n_train_steps'] = int(train_df['step'].iloc[-1])
    if val_df is not None and len(val_df):
        row['val_loss_final'] = float(val_df['val_loss'].iloc[-1])

    # MI for original STATE: results/State/MI/Y_<signal>_<quality>/<seed>/lmi_mutual_information.txt
    mi_root = q_dir / 'results' / 'State' / 'MI'
    if mi_root.is_dir():
        for sig_dir in filter(Path.is_dir, mi_root.iterdir()):
            sm = re.match(r'Y_(.+)_[0-9][0-9_.]*$', sig_dir.name)
            signal = sm.group(1) if sm else sig_dir.name
            f = sig_dir / str(MI_SEED) / 'lmi_mutual_information.txt'
            if f.exists():
                try: row[f'mi_{signal}'] = float(f.read_text().strip())
                except ValueError: pass

    _orig_rows.append(row)
    CURVES[(ORIGINAL_TRIAL_ID, 100_000, float(quality_val))] = {
        'train': train_df, 'val': val_df,
    }

if _orig_rows:
    df = pd.concat([df, pd.DataFrame(_orig_rows)], ignore_index=True)
    df = df.sort_values(['trial_id', 'size', 'quality']).reset_index(drop=True)
    TRIAL_LABEL[ORIGINAL_TRIAL_ID] = f'Original  e256/L3  ({_orig_params/1e6:.1f}M)'
    print(f'Loaded {len(_orig_rows)} original STATE qualities at size=100k')
else:
    print('No original STATE runs found.')


In [ ]:
# Train / validation loss curves: 2 columns (train, val), 1 row per quality.
# Each subplot overlays one line per model-size config, colored by parameter count.
# The original STATE run (trial_id = -1) is overlaid in black/dashed for contrast.
# Smoothing matches 2026-04-17_11-32_plotting_state_loss_curves.ipynb:
#   - drop step==0 (warmup origin distorts the y-scale)
#   - train: rolling(window=200, min_periods=10, center=True).mean()
#   - val:   raw points + connecting line + markers (no smoothing — already sparse)
import matplotlib.pyplot as plt
import numpy as np

SMOOTH_WINDOW = 200
SMOOTH_MIN_PERIODS = 10

qualities = sorted(df['quality'].unique())
trial_ids = sorted(df['trial_id'].unique())
# Color sweep trials with viridis; the Original (tid = -1) is overridden to black.
sweep_ids = [t for t in trial_ids if t != ORIGINAL_TRIAL_ID]
cmap = plt.get_cmap('viridis')
trial_color = {tid: cmap(i / max(1, len(sweep_ids) - 1)) for i, tid in enumerate(sweep_ids)}
trial_color[ORIGINAL_TRIAL_ID] = 'black'
trial_ls = {tid: '-' for tid in trial_ids}
trial_ls[ORIGINAL_TRIAL_ID] = '--'

n_rows = len(qualities)
fig, axes = plt.subplots(n_rows, 2, figsize=(12, 2.6 * n_rows),
                          sharex='col', squeeze=False)

# Track which trial ids we've already labeled so each model size appears
# exactly once in the legend, regardless of which subplot it first shows up in.
labeled = set()

for r, q in enumerate(qualities):
    ax_tr, ax_va = axes[r, 0], axes[r, 1]
    for tid in trial_ids:
        keys = [k for k in CURVES if k[0] == tid and np.isclose(k[2], q)]
        for key in keys:
            curves = CURVES[key]
            tr, va = curves['train'], curves['val']
            color = trial_color[tid]
            ls = trial_ls[tid]
            lw = 1.6 if tid == ORIGINAL_TRIAL_ID else 1.2
            label = TRIAL_LABEL[tid] if tid not in labeled else None
            plotted = False
            if tr is not None and len(tr):
                tr_pos = tr[tr['step'] > 0].sort_values('step')
                if len(tr_pos):
                    smooth = tr_pos['train_loss'].rolling(
                        window=SMOOTH_WINDOW, min_periods=SMOOTH_MIN_PERIODS,
                        center=True).mean()
                    ax_tr.plot(tr_pos['step'].values, smooth.values,
                               color=color, linewidth=lw, linestyle=ls,
                               alpha=0.9, label=label)
                    plotted = True
            if va is not None and len(va):
                va_pos = va[va['step'] > 0].sort_values('step')
                if len(va_pos):
                    ax_va.plot(va_pos['step'].values, va_pos['val_loss'].values,
                               color=color, linewidth=lw + 0.2, linestyle=ls,
                               alpha=0.9, marker='o', markersize=4, label=label)
                    plotted = True
            if plotted and label is not None:
                labeled.add(tid)
    ax_tr.set_ylabel(f'q={q:.4g}\nloss')
    ax_tr.grid(alpha=0.3)
    ax_va.grid(alpha=0.3)

axes[0, 0].set_title('train_loss (rolling mean, w=200)')
axes[0, 1].set_title('val_loss')
axes[-1, 0].set_xlabel('optimizer step')
axes[-1, 1].set_xlabel('optimizer step')

# Collect handles/labels from every subplot so every model size is represented,
# then dedupe by label and order by trial_id for a stable, monotone legend
# (Original pinned to the bottom).
all_handles, all_labels = [], []
for ax in axes.flat:
    h, l = ax.get_legend_handles_labels()
    all_handles += h
    all_labels += l
seen = set()
uniq = [(h, l) for h, l in zip(all_handles, all_labels)
        if not (l in seen or seen.add(l))]
order = {TRIAL_LABEL[tid]: (10**9 if tid == ORIGINAL_TRIAL_ID else tid)
         for tid in trial_ids}
uniq.sort(key=lambda hl: order.get(hl[1], 10**9))
if uniq:
    fig.legend([h for h, _ in uniq], [l for _, l in uniq],
               loc='center left', bbox_to_anchor=(1.0, 0.5),
               fontsize=9, title='model size')
fig.suptitle('STATE PBMC model-size sweep: loss curves per quality (one line per model size)', y=1.0)
plt.tight_layout()
plt.show()


In [ ]:
# MI vs quality, one line per model-size config (overlay).
# Original STATE (trial_id = -1) is rendered black/dashed for contrast.
import matplotlib.pyplot as plt

mi_cols = [c for c in df.columns if c.startswith('mi_')]
if not mi_cols:
    print('No MI columns found — nothing to plot.')
else:
    trial_ids = sorted(df['trial_id'].unique())
    sweep_ids = [t for t in trial_ids if t != ORIGINAL_TRIAL_ID]
    cmap = plt.get_cmap('viridis')
    trial_color = {tid: cmap(i / max(1, len(sweep_ids) - 1)) for i, tid in enumerate(sweep_ids)}
    trial_color[ORIGINAL_TRIAL_ID] = 'black'
    trial_ls = {tid: '-' for tid in trial_ids}
    trial_ls[ORIGINAL_TRIAL_ID] = '--'

    fig, axes = plt.subplots(1, len(mi_cols), figsize=(6.5 * len(mi_cols), 4.5),
                              squeeze=False)
    for j, mi_col in enumerate(mi_cols):
        ax = axes[0, j]
        for tid in trial_ids:
            sub = df[(df['trial_id'] == tid) & df[mi_col].notna()].sort_values('quality')
            if sub.empty:
                continue
            ax.plot(sub['quality'].values, sub[mi_col].values,
                    marker='o', linewidth=1.8, color=trial_color[tid],
                    linestyle=trial_ls[tid], label=TRIAL_LABEL[tid])
        ax.set_xscale('log')
        ax.set_xlabel('quality (downsampling factor)')
        ax.set_ylabel(mi_col.removeprefix('mi_'))
        ax.set_title(f'MI: {mi_col.removeprefix("mi_")} vs quality')
        ax.grid(alpha=0.3)
    # Stable legend order (Original pinned to the bottom).
    handles, labels = axes[0, -1].get_legend_handles_labels()
    order = {TRIAL_LABEL[tid]: (10**9 if tid == ORIGINAL_TRIAL_ID else tid)
             for tid in trial_ids}
    pairs = sorted(zip(handles, labels), key=lambda hl: order.get(hl[1], 10**9))
    if pairs:
        axes[0, -1].legend([h for h, _ in pairs], [l for _, l in pairs],
                           loc='center left', bbox_to_anchor=(1.02, 0.5),
                           fontsize=9, title='model size')
    plt.tight_layout()
    plt.show()
